In [ ]:
from gp import GaussianProcess
import pandas as pd
import jax
import jax.numpy as jnp
import jax.random as jr

from sklearn.preprocessing import MinMaxScaler
import wifiplotting as wp
from wifiplotting import *

from functools import partial

In [ ]:
sequoia = pd.read_csv('../data/sequoia.csv')
sequoia.head()

In [ ]:
sequoia_nonan = sequoia[sequoia.rssi_signal_measured]

scaler = MinMaxScaler()
scaler.fit(pd.DataFrame(np.stack([TL_CORNER[::-1], BR_CORNER[::-1]]), columns=['longitude', 'latitude']))
# scaler.fit(sequoia_nonan[['longitude', 'latitude']])
X_train = pd.DataFrame()
X_train[['longitude', 'latitude']] = scaler.transform(sequoia_nonan[['longitude', 'latitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values

y_train = sequoia_nonan['rssi_sample']

X_train = jnp.array(X_train)
y_train = jnp.array(y_train)

In [ ]:
indoor_mask = X_train[:,2].astype(bool)
outdoor_mask = ~X_train[:,2].astype(bool)

stat = partial(jnp.quantile, q=0.25)
stat1 = partial(jnp.quantile, q=0.75)
stat2 = partial(jnp.quantile, q=0.25)

def m(obs):
    return stat1(y_train[indoor_mask]) * obs[2] + stat2(y_train[outdoor_mask]) * (1-obs[2])

# def m(obs):
#     return stat(y_train)
    
def dist(obs1, obs2):
    '''
    Location distance only
    '''
    return jnp.linalg.norm(obs1[:2] - obs2[:2], ord=2)**2


# def dist(obs1, obs2):
#     '''
#     Location + indoor/outdoor
#     '''
#     return jnp.linalg.norm(obs1 - obs2, ord=2)**2


# def dist(obs1, obs2):
#     '''
#     Location + weighted indoor/outdoor
#     '''
#     loc_dist = jnp.linalg.norm(obs1[:2] - obs2[:2], ord=2)**2
#     inout_dist = jnp.abs(obs1[2] - obs2[2])
#     return loc_dist + (loc_dist  / jnp.sqrt(1)) * inout_dist

scale = min([y_train[indoor_mask].var(), y_train[outdoor_mask].var()])
bandwidth = 1e-2
def K(obs1, obs2):
    d = dist(obs1, obs2)
    return scale * jnp.exp(-d / (2*bandwidth))

In [ ]:
GP = GaussianProcess(m, K)

In [ ]:
GP.fit(X_train, y_train, Sigma_0=jnp.eye(X_train.shape[0])*y_train.var())

In [ ]:
grid_width = 100

x_new = jnp.linspace(0, 1, grid_width)
y_new = jnp.linspace(0, 1, grid_width)
grid_points = jnp.stack(jnp.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)
coord_grid = scaler.inverse_transform(grid_points)
long_new, lat_new = coord_grid.T

osm_context = wp.OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]])

z_new = osm_context.contains_building(long_new, lat_new).reshape(-1,1)

X_new = jnp.concatenate([grid_points, z_new], axis=-1)

from wifiplotting import plot_wifi_heatmap, lonlat_to_world
df_new = pd.DataFrame(X_new, columns=['latitude', 'longitude', 'indoor'])
plot_wifi_heatmap(df_new, new_data=y_mean, aggregate=False)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
candidates=jnp.array([0, 1]).reshape(-1,1)
X_best, y_mean, y_cov = GP.predict(X_new, candidates=None)

vmax = max([y_mean.max(), y_train.max()])
vmin = min([y_mean.min(), y_train.min()])

base_fig, base_ax, osm_metadata = osm_context.generate_base_axis()
# fig = plt.figure(figsize=(8, 6))
wlon, wlat = osm_context.to_world(long_new, lat_new)
base_ax.scatter(wlon, wlat, c=y_mean, cmap="RdYlGn", vmin=vmin, vmax=vmax)
wlon, wlat = osm_context.to_world(sequoia_nonan.longitude, sequoia_nonan.latitude)
s = base_ax.scatter(wlon, wlat, c=y_train, cmap="RdYlGn", vmin=vmin, vmax=vmax)
plt.colorbar(s)
plt.ticklabel_format(style='plain', axis='both', useOffset=False)
plt.tight_layout()

print(f"Data Range: {y_train.min():.3f} to {y_train.max():.3f}")
print(f"Pred Range: {y_mean.min():.3f} to {y_mean.max():.3f}")

In [ ]:
# GP.fit(X_train, y_train, Sigma_0=jnp.eye(X_train.shape[0])*1)
key = jr.PRNGKey(305)
for i in range(50):
    key = GP.gibbs(jr.fold_in(key, i))

In [ ]:
f_hat = GP.f_hat
print(f"f_hat RMSE: {jnp.sqrt(jnp.mean((y_train - f_hat)**2)):.4f}")
print(f"Maximum RSSI Coordinates: ({long_new[f_hat.argmax()]:.5f}, {lat_new[f_hat.argmax()]:.5f}) ")

In [ ]:
jnp.sqrt(jnp.mean((y_train - y_train.mean())**2))